<a href="https://colab.research.google.com/github/nicolasramirezperilla/DataWave-Project/blob/master/Consolidado_BBDD_CR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##1) Instalar librerias y conexión al servidor.

In [ ]:
# Importing libraries
from google.colab import auth
from google.colab import files
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil import parser  # Import dateutil.parser for automatic date parsing

# Formatting for viewing tables
from google.colab import data_table
data_table.enable_dataframe_formatter()

# Authenticating Google Sheets
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

from gspread_dataframe import get_as_dataframe
import pandas as pd

gc = gspread.authorize(creds)
client = gspread.authorize(creds)
import locale

##2) Descargar información, definir parametros y transformación.

In [ ]:
# Paso 1: Función para procesar un archivo
def procesar_archivo(nombre_archivo):
    # Paso 1.1: Abrir el archivo de Google Sheets por su nombre
    spreadsheet = client.open(nombre_archivo)  # Cambia el nombre del archivo dinámicamente
    sheet = spreadsheet.worksheet("P&L")  # Acceder a la hoja llamada 'P&L'

    # Paso 1.2: Obtener todos los registros sin considerar la primera fila como cabeceras
    data_raw = sheet.get_all_values()

    # Paso 1.3: Filtrar las columnas desde la N (índice 13) hasta la AZ (índice 51)
    data_filtered = [row[12:52] for row in data_raw]  # Filtra las columnas de la N a la AZ

    # Paso 1.4: Crear tres DataFrames con los rangos de filas especificados

    # DataFrame 1: Fila 7 a 47 (índices 6 a 46 en Python)
    df1_raw = data_filtered[6:47]  # Extraemos las filas necesarias
    df1_headers = df1_raw[0]  # La primera fila dentro del rango será la cabecera
    df1 = pd.DataFrame(df1_raw[1:], columns=df1_headers)  # Asignar las cabeceras correctas

    # DataFrame 2: Fila 53 a 93 (índices 52 a 92 en Python)
    df2_raw = data_filtered[52:93]  # Extraemos las filas necesarias
    df2_headers = df2_raw[0]  # La primera fila dentro del rango será la cabecera
    df2 = pd.DataFrame(df2_raw[1:], columns=df2_headers)  # Asignar las cabeceras correctas

    # DataFrame 3: Fila 99 a 139 (índices 98 a 138 en Python)
    df3_raw = data_filtered[98:139]  # Extraemos las filas necesarias
    df3_headers = df3_raw[0]  # La primera fila dentro del rango será la cabecera
    df3 = pd.DataFrame(df3_raw[1:], columns=df3_headers)  # Asignar las cabeceras correctas

    # Paso 1.5: Función para transformar un DataFrame
    def reestructurar_df(df):
        # Reshape: pivotamos el DataFrame para que las fechas sean la columna 'Fecha'
        df_reshaped = pd.melt(df, id_vars=[df.columns[0]], var_name='Fecha', value_name='Real')

        # Renombramos las columnas
        df_reshaped.rename(columns={df.columns[0]: 'Rubro'}, inplace=True)

        return df_reshaped

    # Paso 1.6: Transformar los tres DataFrames
    df1_reshaped = reestructurar_df(df1)
    df2_reshaped = reestructurar_df(df2)
    df3_reshaped = reestructurar_df(df3)

    # Paso 1.7: Añadir la columna 'Filial' con el nombre del archivo

    df1_reshaped['Filial'] = nombre_archivo
    df2_reshaped['Filial'] = nombre_archivo
    df3_reshaped['Filial'] = nombre_archivo

    # Eliminar los espacios múltiples y dejar solo un espacio
    df1_reshaped['Rubro'] = df1_reshaped['Rubro'].str.replace(r'\s+', ' ', regex=True)
    df2_reshaped['Rubro'] = df2_reshaped['Rubro'].str.replace(r'\s+', ' ', regex=True)
    df3_reshaped['Rubro'] = df3_reshaped['Rubro'].str.replace(r'\s+', ' ', regex=True)

    # Paso 1.8: Realizar el cruce de df1_reshaped y df2_reshaped por 'Rubro' y 'Fecha'
    df1_df2_merged = pd.merge(df1_reshaped, df2_reshaped, on=['Rubro', 'Fecha','Filial'], how='left', suffixes=('_df1', '_df2'))

    # Paso 1.9: Realizar el cruce del resultado anterior con df3_reshaped por 'Rubro' y 'Fecha'
    df_merged = pd.merge(df1_df2_merged, df3_reshaped, on=['Rubro', 'Fecha','Filial'], how='left', suffixes=('_df2', '_df3'))

    # Paso 1.10: Renombrar las columnas
    df_merged.rename(columns={
        'Real_df1': 'Real',
        'Real_df2': 'Proyeccion',
        'Real': 'Presupuesto'
    }, inplace=True)

    return df_merged

# Paso 2: Lista de nombres de archivos (puedes poner los nombres de los archivos aquí)
nombres_archivos = ["02 Fiduciaria CR","01 Banco CR", "03 Valores CR", "04 CSF CR", "05 E9 CR", "06 Openpay CR","07 Movistar CR", "08 EFAN CR",
                    "09 Complemento CR","10 EFAN Seguros CR","11 Seguros CR","12 Eliminación Movistar CR"]  # Añade más nombres de archivo

# Paso 3: Iterar sobre los archivos y procesarlos
resultados = []

for archivo in nombres_archivos:
    df_resultado = procesar_archivo(archivo)
    resultados.append(df_resultado)

# Paso 4: Concatenar todos los DataFrames resultantes
df_final = pd.concat(resultados, ignore_index=True)

# Paso 4.1: Eliminar las columnas cuyos nombres contienen '_'
df_final = df_final.loc[:, ~df_final.columns.str.contains('_')]

# Paso 4.2: Reorganizar las columnas para que 'Filial' sea la primera
columnas = ['Filial'] + [col for col in df_final.columns if col != 'Filial']
df_final = df_final[columnas]

##3) Realizar los cálculos de las métricas principales.

In [ ]:
#Calculos

# Paso 0: Cambiar Tipo Numero, Tipo Fecha, Mes & Año
df_final['Real'] = pd.to_numeric(df_final['Real'].astype(str).str.replace(',', '', regex=True), errors='coerce')
df_final['Presupuesto'] = pd.to_numeric(df_final['Presupuesto'].astype(str).str.replace(',', '', regex=True), errors='coerce')
df_final['Proyeccion'] = pd.to_numeric(df_final['Proyeccion'].astype(str).str.replace(',', '', regex=True), errors='coerce')

locale.setlocale(locale.LC_TIME, 'en_US.UTF-8')  # Cambiar a inglés (USA)
df_final['Fecha'] = pd.to_datetime(df_final['Fecha'], format='%b-%y')
df_final['Año'] = df_final['Fecha'].dt.year
meses_nombre = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril',
    5: 'Mayo', 6: 'Junio', 7: 'Julio', 8: 'Agosto',
    9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}
df_final['Mes'] = df_final['Fecha'].dt.month.astype(str).str.zfill(2) + ' ' + df_final['Fecha'].dt.month.map(meses_nombre)

fecha_index = df_final.columns.get_loc('Fecha')  # Obtener la posición de 'Fecha'
df_final.insert(fecha_index + 1, 'Año', df_final.pop('Año'))
df_final.insert(fecha_index + 2, 'Mes', df_final.pop('Mes'))

# Paso 1: Acumulados
df_final['Real_Acum_Año'] = df_final.groupby(['Filial', 'Rubro', 'Año'])['Real'].cumsum()
df_final['Ppto_Acum_Año'] = df_final.groupby(['Filial', 'Rubro', 'Año'])['Presupuesto'].cumsum()
df_final['Proy_Acum_Año'] = df_final.groupby(['Filial', 'Rubro', 'Año'])['Proyeccion'].cumsum()

# Paso 2: Valores Last Year/Last Month

# Ordenar por Filial, Rubro y Fecha para asegurar la secuencia correcta
df_final = df_final.sort_values(by=['Filial', 'Rubro', 'Fecha'], ascending=[True, True, True])

# Calcular Real_Año_Anterior: Restar 12 meses para obtener el año anterior
df_final['Real_Año_Anterior'] = df_final.groupby(['Filial', 'Rubro'])['Real'].shift(12)
df_final['Real_Año_12M'] = df_final.groupby(['Filial', 'Rubro'])['Real_Acum_Año'].shift(12)

# Calcular Real_Mes_Anterior: Tomar el mes anterior
df_final['Real_Mes_Anterior'] = df_final.groupby(['Filial', 'Rubro'])['Real'].shift(1)

##4) Crear columnas y modificar tipos de formato.

In [ ]:
#Formato
# Paso 1: Reemplazos Columna 'Filial'& 'Rubro', y Nan
df_final = df_final.fillna(0)
df_final['Filial'] = df_final['Filial'].str.replace('CR', '', regex=True) \
                           .str.replace(r'\d+', '', regex=True) \
                           .str.strip()
df_final['Rubro'] = df_final['Rubro'].str.strip()

#Paso 3: Cuenta Principal

rubros_principales_actualizados  = [
    "Margen de Intereses",
    "Comisiones Netas",
    "ROFs",
    "Resto Ingresos Netos Ordinarios",
    "Margen Bruto",
    "Gastos de Explotación",
    "Gastos de Personal",
    "Gastos Generales",
    "Tributos",
    "Amortizaciones",
    "Margen Neto",
    "Saneamiento Crediticio",
    "Pérdida Deterioro Resto de Activos",
    "Dotaciones a Provisiones",
    "Resultados de Explotación",
    "Resto de Resultados No Ordinarios",
    "BAI",
    "Impuesto Sociedades",
    "BDI",
    "Intereses minoritarios",
    "Resultado atribuido"]

df_final["Cuenta Principal"] = df_final["Rubro"].isin(rubros_principales_actualizados )
rubro_index = df_final.columns.get_loc('Rubro')  # Obtener la posición de 'Rubro'
df_final.insert(rubro_index + 1, 'Cuenta Principal', df_final.pop('Cuenta Principal'))

# Paso 4: Columnas Marca Fecha Ult Mes y Ult Año

# Filtrar filas donde Real no sea 0
filtered_df = df_final[df_final['Real'] != 0]

# Obtener la última fecha (Fecha máxima) para cada filial donde Real no es 0
ultima_fecha = (
    filtered_df.groupby('Filial')
    .agg(Ultima_Fecha=('Fecha', 'max'))
    .reset_index()
)


# Unir las últimas fechas al DataFrame original
df_final = df_final.merge(ultima_fecha, on='Filial', how='left')

# Crear la columna "Marca Mes" (True si coincide con la última fecha)
df_final['Marca Mes'] = df_final['Fecha'] == df_final['Ultima_Fecha']

# Crear la columna "Marca Año" (True si está dentro del rango de un año antes hasta la última fecha)
df_final['Marca Año'] = (df_final['Fecha'] >= (df_final['Ultima_Fecha'] - pd.DateOffset(years=1))) & (df_final['Fecha'] <= df_final['Ultima_Fecha'])

#Paso 5: Crear Real o Proyección
df_final['Real o Proyeccion'] = np.where(
    df_final['Fecha'] > df_final['Ultima_Fecha'],
    df_final['Proyeccion'],
    df_final['Real']
)

df_final['Real o Proyeccion Acum.'] = df_final.groupby(['Filial', 'Rubro', 'Año'])['Real o Proyeccion'].cumsum()

ppto_index = df_final.columns.get_loc('Real_Mes_Anterior')  # Obtener la posición de 'Real_Mes_Anterior'
df_final.insert(ppto_index + 1, 'Real o Proyeccion', df_final.pop('Real o Proyeccion'))

pptoacum_index = df_final.columns.get_loc('Real o Proyeccion')  # Obtener la posición de 'Real o Proyeccion'
df_final.insert(pptoacum_index + 1, 'Real o Proyeccion Acum.', df_final.pop('Real o Proyeccion Acum.'))

# Paso 8:Orden Cuenta
# Definir el diccionario de mapeo
mapeo_rubro = {
    "Margen de Intereses": 1,
    "Comisiones Netas": 2,
    "ROFs": 3,
    "Resto Ingresos Netos Ordinarios": 4,
    "Margen Bruto": 5,
    "Gastos de Explotación": 6,
    "Gastos de Personal": 7,
    "Gastos Generales": 8,
    "Tributos": 9,
    "Amortizaciones": 10,
    "Margen Neto": 11,
    "Saneamiento Crediticio": 12,
    "Pérdida Deterioro Resto de Activos": 13,
    "Dotaciones a Provisiones": 14,
    "Resultados de Explotación": 15,
    "Resto de Resultados No Ordinarios": 16,
    "BAI": 17,
    "Impuesto Sociedades": 18,
    "BDI": 19,
    "Intereses minoritarios": 20,
    "Resultado atribuido": 21
}

# Aplicar el mapeo a la columna 'Rubro'
df_final['Orden Rubro'] = df_final['Rubro'].map(mapeo_rubro)

antesrubro_index = df_final.columns.get_loc('Rubro')  # Obtener la antes posición de 'Rubro'
df_final.insert(antesrubro_index, 'Orden Rubro', df_final.pop('Orden Rubro'))

# Paso 7:Formato Columna Fecha
# Convertir las fechas al formato YYYY-MM
df_final['Fecha Corta'] = df_final['Fecha'].dt.to_period('M').astype(str)
fechados_index = df_final.columns.get_loc('Fecha')  # Obtener la antes posición de 'Rubro'
df_final.insert(fechados_index, 'Fecha Corta', df_final.pop('Fecha Corta'))

df_final['Fecha'] = df_final['Fecha'].astype(str)

df_final['Ultima_Fecha'] = df_final['Ultima_Fecha'].dt.to_period('M').astype(str)

##5) Actualizar hojas de cálculo en Google Sheets

In [ ]:
input_workbook_name = 'CRBBDD'
output_sheet_df_total = gc.open(input_workbook_name).worksheet('BBDD')
output_sheet_df_total.clear()
output_sheet_df_total.update([df_final.columns.values.tolist()] + df_final.fillna(0).values.tolist())

{'spreadsheetId': '1IVB587b2xXf_t6TRvF0w8jAxV0MjdEninkor69wVmeE',
 'updatedRange': 'BBDD!A1:V18721',
 'updatedRows': 18721,
 'updatedColumns': 22,
 'updatedCells': 411862}